# CyberSec-FT-LLM - Comprehensive Evaluation
This notebook loads your fine-tuned Unsloth model from Google Drive and evaluates its performance on the `test.jsonl` dataset using quantitative NLP metrics:

* **Perplexity** (Language modeling confidence)
* **ROUGE** (Recall-Oriented Summarization)
* **BLEU / SacreBLEU** (Precision-based translation/generation)
* **METEOR** (Semantic alignment)
* **BERTScore** (Contextual embedding similarity)

In [ ]:
# Cell 1 - Install Dependencies
!pip install -q unsloth transformers datasets evaluate rouge_score sacrebleu bert_score meteor nltk accelerate
print("Dependencies Installed!")

In [ ]:
# Cell 2 - Mount Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Cell 3 - Load Test Dataset
import json
import os

DATASET_PATH = '/content/drive/MyDrive/CyberSec-FT-LLM/dataset/test.jsonl'

def load_jsonl(path, max_samples=100):
    rows = []
    if not os.path.exists(path):
        print(f"Dataset not found at {path}. Please upload it.")
        return rows
        
    for i, line in enumerate(open(path, encoding='utf-8', errors='replace')):
        if i >= max_samples: break
        line = line.replace('\x00', '').strip()
        if line:
            try: rows.append(json.loads(line))
            except Exception: pass
    return rows

print("Loading test data (limiting to 100 samples for evaluation speed)...")
test_data = load_jsonl(DATASET_PATH, max_samples=100)
print(f"Loaded {len(test_data)} test samples.")

In [ ]:
# Cell 4 - Load Fine-Tuned Model via Unsloth
from unsloth import FastLanguageModel
import torch
import warnings
warnings.filterwarnings("ignore")

from transformers import logging as hf_logging
hf_logging.set_verbosity_error()

# Point this to your latest checkpoint or adapter folder!
ADAPTER_PATH = '/content/drive/MyDrive/CyberSec-FT-LLM/models/adapter'

print(f"Loading adapter from: {ADAPTER_PATH}")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=ADAPTER_PATH,
    max_seq_length=1024,
    dtype=None,
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model) # Enable native 2x faster inference
print("Model ready for inference!")

In [ ]:
# Cell 5 - Generate Predictions
from tqdm.auto import tqdm

def format_prompt(sample):
    u = sample.get('instruction', '')
    if sample.get('input'): 
        u = u + '\n\n' + sample.get('input', '')
    return tokenizer.apply_chat_template(
        [{'role': 'user', 'content': u}],
        tokenize=False, 
        add_generation_prompt=True
    )

references = []
predictions = []
prompts = []

print("Generating predictions... This will take a few minutes.")
for sample in tqdm(test_data):
    prompt = format_prompt(sample)
    ref = sample.get('output', '')
    
    inputs = tokenizer([prompt], return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=256, use_cache=True, pad_token_id=tokenizer.eos_token_id)
        
    pred = tokenizer.batch_decode(outputs[:, inputs.input_ids.shape[1]:], skip_special_tokens=True)[0]
    
    prompts.append(prompt)
    references.append(ref)
    predictions.append(pred)

print(f"Generated {len(predictions)} predictions.")

In [ ]:
# Cell 6 - Calculate Perplexity
import math

def compute_perplexity(texts):
    model.eval()
    total_loss, count = 0.0, 0
    with torch.no_grad():
        for text in tqdm(texts, desc="Calculating PPL"):
            enc = tokenizer(text, return_tensors="pt", max_length=1024, truncation=True).to(model.device)
            labels = enc["input_ids"].clone()
            out = model(**enc, labels=labels)
            total_loss += out.loss.item()
            count += 1
    avg_loss = total_loss / max(count, 1)
    return math.exp(avg_loss)

# We calculate perplexity on the full expected text (instruction + output)
full_texts = [p + r for p, r in zip(prompts, references)]
ppl = compute_perplexity(full_texts)

print(f"\n--- PERPLEXITY ---")
print(f"Model Perplexity: {ppl:.2f}")
print("(Lower is better. Language modeling confidence on the security text.)")

In [ ]:
# Cell 7 - ROUGE Score (Recall-Oriented Understudy for Gisting Evaluation)
import evaluate
rouge = evaluate.load("rouge")

results = rouge.compute(predictions=predictions, references=references)

print("--- ROUGE SCORES ---")
print(f"ROUGE-1 (Unigram overlap): {results['rouge1']:.4f}")
print(f"ROUGE-2 (Bigram overlap):  {results['rouge2']:.4f}")
print(f"ROUGE-L (Longest seq overlap): {results['rougeL']:.4f}")

In [ ]:
# Cell 8 - BLEU Score (SacreBLEU)
sacrebleu = evaluate.load("sacrebleu")

# SacreBLEU expects references as a list of lists: [[ref1_for_pred1], [ref1_for_pred2]]
refs_for_bleu = [[ref] for ref in references]
bleu_results = sacrebleu.compute(predictions=predictions, references=refs_for_bleu)

print("--- BLEU SCORE ---")
print(f"BLEU Score: {bleu_results['score']:.2f}")
print("(Standard translation/generation metric)")

In [ ]:
# Cell 9 - METEOR Score (Metric for Evaluation of Translation with Explicit ORdering)
import nltk
nltk.download('wordnet', quiet=True)
nltk.download('punkt_tab', quiet=True)
meteor = evaluate.load("meteor")

meteor_results = meteor.compute(predictions=predictions, references=references)

print("--- METEOR SCORE ---")
print(f"METEOR Score: {meteor_results['meteor']:.4f}")
print("(Aligns synonyms and stems, heavily penalizing bad word order)")

In [ ]:
# Cell 10 - BERTScore (Contextual Embedding Similarity)
bertscore = evaluate.load("bertscore")

print("Calculating BERTScore... (This downloads a RoBERTa eval model to compute semantic similarities)")
bert_results = bertscore.compute(predictions=predictions, references=references, lang="en")

import numpy as np
f1_mean = np.mean(bert_results['f1'])
precision_mean = np.mean(bert_results['precision'])
recall_mean = np.mean(bert_results['recall'])

print("\n--- BERTSCORE ---")
print(f"Semantic Precision: {precision_mean:.4f}")
print(f"Semantic Recall:    {recall_mean:.4f}")
print(f"Semantic F1 Score:  {f1_mean:.4f}")
print("(How closely the meaning matches, ignoring exact keyword matching)")